In [1]:
from config import (
    CENTRALS_PATH, SATELLITES_PATH, JACKKNIFE_CATALOG_PATH,
    WP_FIDUCIAL_PATH, WP_JACKKNIFE_PATH, NJN, LOS, MR_THRESH, SEED,
)
from hod_pipeline import (
    load_halos, write_fits_bolshoi, get_hod_params, generate_mock_rsd,
    compute_wp, save_mock_fits, load_mock_fits,
    save_jackknife_catalog, load_jackknife_catalog,
    save_wp_fiducial, save_wp_jackknife,
)
from jackknife import JackKnife
from jackknife_covariance import JackknifeCovariance
import numpy as np

# 1. halos + fiducial mock
halos, LBOX, OMEGA_M, HUBBLE, REDSHIFT = load_halos()
write_fits_bolshoi(halos)
BASE_HOD = get_hod_params(MR_THRESH)
mock_base = generate_mock_rsd(BASE_HOD, seed=SEED, rsd=True, Lbox=LBOX, Omega_m=OMEGA_M)

# 2. save + reload
save_mock_fits(mock_base, CENTRALS_PATH, SATELLITES_PATH)
mock_loaded = load_mock_fits(CENTRALS_PATH, SATELLITES_PATH)

# 3. full-sample wp
rp_base, wp_base = compute_wp(mock_loaded, lbox=LBOX)
save_wp_fiducial(rp_base, wp_base, WP_FIDUCIAL_PATH)

# --- CHECK OUTPUTS HERE before proceeding to the expensive 100-region loop ---

# 4. jackknife region assignment -> save
positions = np.column_stack([mock_loaded['x'], mock_loaded['y'], mock_loaded['z']])
jk = JackKnife(positions, LBOX)
data, _ = jk.add_jackknife_regions(njn=NJN, los=LOS)
save_jackknife_catalog(data, JACKKNIFE_CATALOG_PATH)

# 5. READ BACK the saved catalogue — covariance consumes this, not a fresh assignment
tagged_data = load_jackknife_catalog(JACKKNIFE_CATALOG_PATH)

def position_to_mock(pos):
    return {'x': pos[:, 0], 'y': pos[:, 1], 'z': pos[:, 2]}

jkcov = JackknifeCovariance(tagged_data, LBOX, compute_wp, njn=NJN)
rp_jk, wp_mean, cov, diag_err = jkcov.compute(position_to_mock)
jkcov.sanity_check(rp_base, wp_base)

save_wp_jackknife(rp_jk, wp_mean, cov, diag_err, WP_JACKKNIFE_PATH,
                  njn=NJN, los=LOS, Mr_thresh=MR_THRESH, seed=SEED, rsd=True)

Snapshot Reader:
... preparing to read file: /data/project/hpc2502016/data/sims/sahyadri/default2048/r1/snapdir_082/snapshot_082
... loaded header and parameters
... preparing halo data
... using file: sahyadri/default2048/r1/out_82.trees
... ... using mass definition m200b > 8.108e+10 Msun/h
... ... only relaxed objects retained with 0.50 < 2T/|U| < 1.50
... ... discarding subhalos
... ... kept 343275 of 10493239 objects in catalog
Wrote 343275 halos -> /mnt/home/project/cgowari.aditya/sahyadri-codes/sahyadri_r1_snap100_bolshoi_schema.fits
Saved 210,196 centrals -> results/centrals.fits
Saved 82,396 satellites -> results/satellites.fits
Loaded 210,196 centrals + 82,396 satellites = 292,592 galaxies
Saved fiducial wp -> results/wp_fiducial.npz
Saved 292,592 rows, 100 regions -> results/gpos_jackknife_njn100_los1.fits
Loaded 292,592 rows, 100 regions <- results/gpos_jackknife_njn100_los1.fits
  ... region 20/100 done
  ... region 40/100 done
  ... region 60/100 done
  ... region 80/100 

PosixPath('results/wp_jackknife_njn100.npz')

In [2]:
from config import SEED
from hod_pipeline import get_hod_params
from derivative_pipeline import run_all_derivatives

BASE_HOD = get_hod_params(MR_THRESH)
derivs = run_all_derivatives(BASE_HOD, seed=SEED, Lbox=LBOX, Omega_m=OMEGA_M)

# e.g. inspect one result:
rp, dwp_Mcut_5pct = derivs['Mcut'][0.050]


── Mcut  δ = ±5.0% ──
  Mcut: 1.42569e+11 -> +5.0%: 1.49697e+11  |  -5.0%: 1.3544e+11
  N_gal: +5.0% = 283,636   -5.0% = 302,556
  Saved -> Mcut_plus5.npz, Mcut_minus5.npz

── Mcut  δ = ±2.5% ──
  Mcut: 1.42569e+11 -> +2.5%: 1.46133e+11  |  -2.5%: 1.39005e+11
  N_gal: +2.5% = 287,988   -2.5% = 297,437
  Saved -> Mcut_plus2p5.npz, Mcut_minus2p5.npz

── M1  δ = ±5.0% ──
  M1: 2.83813e+12 -> +5.0%: 2.98004e+12  |  -5.0%: 2.69623e+12
  N_gal: +5.0% = 287,603   -5.0% = 298,139
  Saved -> M1_plus5.npz, M1_minus5.npz

── M1  δ = ±2.5% ──
  M1: 2.83813e+12 -> +2.5%: 2.90909e+12  |  -2.5%: 2.76718e+12
  N_gal: +2.5% = 290,128   -2.5% = 295,305
  Saved -> M1_plus2p5.npz, M1_minus2p5.npz

── M0  δ = ±5.0% ──
  M0: 1.2764e+07 -> +5.0%: 1.34022e+07  |  -5.0%: 1.21258e+07
  N_gal: +5.0% = 292,592   -5.0% = 292,592
  Saved -> M0_plus5.npz, M0_minus5.npz

── M0  δ = ±2.5% ──
  M0: 1.2764e+07 -> +2.5%: 1.30831e+07  |  -2.5%: 1.24449e+07
  N_gal: +2.5% = 292,592   -2.5% = 292,592
  Saved -> M0_plus2p5.